# SW-16 — Proof-Carrying Ontologies : le certificat d'inférence (jumeau C#)

**Navigation** : [Index](README.md) | [<< SW-15](SW-15-Coup-Argumentatif-Python.ipynb) · jumeau : [Python](SW-16-ProofCarryingOntologies-Python.ipynb)

**Concept** : SW-13 a comparé les raisonneurs OWL (owlrl, HermiT, reasonable) — tous répondent *oui* ou *non*, aucun n'explique *pourquoi il le croit*. Ce notebook introduit le **proof-carrying inference** : chaque conclusion s'accompagne d'un **certificat** qu'un checker **indépendant** accepte ou rejette. Le concept est incarné par la plateforme [open-ontologies](https://github.com/fabio-rovai/open-ontologies) (Fabio Rovai, MIT, moteur Rust + checkers Lean 4 prouvés sound, préprint [arXiv:2605.09184](https://arxiv.org/abs/2605.09184)), dont le README s'ouvre sur une phrase qui vaut un cours :

> *Ask a reasoner why it believes something and it will tell you to trust it. This one hands you the proof, and refuses a fake one.*
> (« Demandez à un raisonneur pourquoi il croit quelque chose, il vous répondra de lui faire confiance. Celui-ci vous tend la preuve, et refuse une preuve forgée. »)

**Spécificité du jumeau C#** : le moteur jouet et le checker sont écrits sur la **seule base de classes .NET** (aucun paquet NuGet — `System.Text.Json` pour le certificat, LINQ pour le chaînage), et la jambe réelle invoque le binaire open-ontologies épinglé par `System.Diagnostics.Process` — la voie *native-cli* : le vrai moteur, en vrai processus, empreinte SHA256 vérifiée avant toute invocation.

**Plan** :

1. **Contexte** — le gap de SW-13 et la table fil rouge « avec une preuve / sans preuve » ;
2. **L'ontologie jouet et le moteur Horn naïf** — forward chaining qui enregistre *pourquoi* ;
3. **Le certificat d'inférence** — un artefact transportable, vérifiable par un tiers ;
4. **Le checker indépendant** — vérifie, ne re-déduit pas ; verdicts et rejets nommés ;
5. **Quatre forges rejetées** — le résultat falsifiable ;
6. **La jambe réelle** — le binaire open-ontologies v1.4.0 sur l'ontologie pizza de la série, et notre checker jouet qui vérifie son vrai certificat ;
7. **Ce que le certificat NE prouve PAS** — les limites load-bearing, écrites ;
8. **L'honnêteté documentaire** — la dérive documentaire d'un dépôt tiers comme cas d'étude ;
9. **Table fil rouge finale et cross-links** — vericode, Z3, Tweety, Lean.

### Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. expliquer pourquoi une conclusion de raisonneur sans preuve transportable est un acte de confiance, pas un fait vérifiable ;
2. implémenter en C# un moteur Horn naïf qui **émet** une étape de dérivation par inférence ;
3. écrire un **checker indépendant** de certificats (unification, disponibilité ordonnée des prémisses, auto-support) qui rejette nommément les certificats forgés ;
4. distinguer les verdicts `entailed` et `entailed_under_supplied_rules` — une règle que *vous* avez écrite est une supposition, jamais un fait ;
5. invoquer le vrai binaire open-ontologies depuis .NET (`Process`, tag figé, SHA256 vérifié) et vérifier son certificat `oo-cert/1` ;
6. énoncer ce qu'un certificat **ne prouve pas** : provenance des assertions, moteurs partageant le même algorithme, théorème cité mais non machine-checké.

### Prérequis

- **SW-02** (triplets RDF), **SW-07** (OWL), **SW-13** (raisonneurs) — utiles, pas bloquants ;
- .NET 9 SDK et le kernel .NET Interactive (`dotnet tool install --global Microsoft.dotnet-interactive`) ; aucune clé API requise ;
- la section 6 télécharge un binaire de ~58 Mo au premier passage (ensuite mis en cache local).

### Durée estimée : 50 minutes

In [1]:
// Outillage de ce notebook : la base de classes (BCL) .NET uniquement pour le moteur jouet.
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Reflection;
using System.Runtime.InteropServices;
using System.Security.Cryptography;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;

Console.WriteLine($".NET : {RuntimeInformation.FrameworkDescription}");
Console.WriteLine("Moteur jouet : BCL .NET (System.Text.Json, LINQ) — aucun paquet NuGet.");
Console.WriteLine("Jambe réelle (section 6) : binaire open-ontologies, auto-installé si absent.");

.NET : .NET 10.0.15


Moteur jouet : BCL .NET (System.Text.Json, LINQ) — aucun paquet NuGet.


Jambe réelle (section 6) : binaire open-ontologies, auto-installé si absent.


## 1. Contexte — le gap de SW-13

[SW-13](SW-13-Reasoners-Python.ipynb) a benchmarké owlrl, HermiT (via OWLReady2), reasonable : temps d'exécution, nombre de triplets inférés, facilité d'installation. Ce que le benchmark ne pouvait pas mesurer, c'est la **nature de la réponse**. Interrogez owlrl sur « margherita est-elle un aliment ? », il matérialise le triplet — et si vous lui demandez *pourquoi*, la réponse est dans le code du moteur, pas dans un artefact que vous pourriez transmettre à un tiers. Un grep de `certificate`, `proof` ou `justif` dans SW-13 rend zéro résultat : le concept même de preuve transportable est absent — c'est le gap que ce notebook comble.

L'idée a un précédent célèbre côté langages : le **proof-carrying code** (Necula, 1997), où un compilateur non fiable accompagne chaque binaire d'une preuve qu'un checker minuscule — et donc auditable — valide avant d'exécuter. open-ontologies transpose ce geste au Web Sémantique : le **raisonneur** (Rust, ou un moteur C# pur) n'est *pas* fiable, mais chaque inférence s'accompagne d'un certificat qu'un checker écrit en Lean 4 — dont la correction est un **théorème machine-checké** (`OOCert.certificate_sound`) — accepte ou rejette :

```mermaid
flowchart LR
    E["Moteur non fiable<br/>Rust, ou le jouet C# de ce notebook"] -->|certificat| C["Checker indépendant<br/>Lean 4 (théorème), ou jouet C# (procédure)"]
    C -->|règles intégrées| A["entailed"]
    C -->|règles fournies| B["entailed_under_supplied_rules"]
    C -->|certificat forgé| X["rejeté, raison nommée"]
```

### La table fil rouge — avec une preuve, sans une preuve

Cette table reviendra en fin de notebook (section 9), enrichie par ce que nous aurons mesuré. Elle est l'ossature du cours :

| Question | Sans preuve (statut quo SW-13) | Avec un certificat (ce notebook) |
|---|---|---|
| Pourquoi croire à la conclusion ? | « le raisonneur l'a dit » | une suite d'étapes, chacune relue |
| Un tiers peut vérifier ? | non — il doit ré-exécuter le même moteur | oui — le fichier suffit, le moteur n'intervient plus |
| Une conclusion peut-elle être forgée ? | indétectable | rejet **nommé** (premisse absente, auto-support, liaison incohérente…) |
| Une règle métier change-t-elle le statut ? | non distingué | oui — `entailed_under_supplied_rules` ≠ `entailed` |
| Que prouve le « non » ? | rien (opinion d'oracle) | rien non plus — c'est une limite écrite (section 7) |

## 2. L'ontologie jouet et le moteur Horn naïf

Le domaine rejoint la fixture pizza de la série (`data/pizza.owl`, utilisée par SW-7 et SW-13) en miniature : une pizzeria, des ingrédients, des allergènes. Neuf triplets assertés, quatre règles Horn sur motifs de triplets — trois règles RDFS « intégrées » (`rdfs9`, `rdfs11`, `rdfs3`) et une règle métier écrite par nous (`allergenPropagation` : si une pizza contient un ingrédient qui porte un allergène, la pizza porte l'allergène).

**Notation** : un triplet est un tuple `(sujet, prédicat, objet)` d'IRIs ; une règle a des prémisses et une conclusion qui sont des *patrons* de triplets, où les variables commencent par `?`. Exemple `rdfs9` — la propagation de type par sous-classe :

$$
\underbrace{(?x,\ \texttt{type},\ ?c)}_{\text{prémisse 1}},\ \underbrace{(?c,\ \texttt{subClassOf},\ ?d)}_{\text{prémisse 2}} \;\Longrightarrow\; (?x,\ \texttt{type},\ ?d)
$$

La distinction **règle intégrée / règle fournie** n'est pas cosmétique : nous verrons au checker qu'elle change le *verdict* — c'est exactement la sémantique d'open-ontologies, où une règle que vous avez écrite est une **supposition que le certificat transporte**, jamais un fait qu'il établit.

In [2]:
// L'ontologie jouet : faits assertés + table de règles Horn.
const string RdfType = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type";
const string RdfsSubClass = "http://www.w3.org/2000/01/rdf-schema#subClassOf";
const string RdfsRange = "http://www.w3.org/2000/01/rdf-schema#range";
const string Ex = "http://example.org/pizza#";

static string Q(string local) => Ex + local;
static string Court(string iri) => iri.Split('#').Last().Split('/').Last();

var asserted = new List<(string S, string P, string O)> {
    // instances
    (Q("margherita"), RdfType, Q("NamedPizza")),
    (Q("margherita"), Q("containsIngredient"), Q("mozzarella")),
    (Q("mozzarella"), Q("hasAllergen"), Q("milk")),
    (Q("mozzarella"), RdfType, Q("DairyProduct")),
    // hiérarchie de classes
    (Q("NamedPizza"), RdfsSubClass, Q("Pizza")),
    (Q("Pizza"), RdfsSubClass, Q("Food")),
    (Q("Food"), RdfsSubClass, Q("Consumable")),
    (Q("DairyProduct"), RdfsSubClass, Q("AllergenSource")),
    // schéma : portée de hasAllergen
    (Q("hasAllergen"), RdfsRange, Q("Allergen")),
};

var rules = new Dictionary<string, (string[][] Premises, string[] Conclusion)> {
    ["rdfs9"] = (new[] { new[] { "?x", RdfType, "?c" }, new[] { "?c", RdfsSubClass, "?d" } },
                 new[] { "?x", RdfType, "?d" }),
    ["rdfs11"] = (new[] { new[] { "?a", RdfsSubClass, "?b" }, new[] { "?b", RdfsSubClass, "?c" } },
                  new[] { "?a", RdfsSubClass, "?c" }),
    ["rdfs3"] = (new[] { new[] { "?p", RdfsRange, "?c" }, new[] { "?x", "?p", "?y" } },
                 new[] { "?y", RdfType, "?c" }),
    ["allergenPropagation"] = (new[] { new[] { "?pizza", Q("containsIngredient"), "?ing" },
                                       new[] { "?ing", Q("hasAllergen"), "?all" } },
                               new[] { "?pizza", Q("hasAllergen"), "?all" }),
};
var builtinRules = new HashSet<string> { "rdfs9", "rdfs11", "rdfs3" };

Console.WriteLine($"Faits assertés   : {asserted.Count}");
Console.WriteLine($"Règles           : {rules.Count}  (intégrées : {string.Join(", ", builtinRules.OrderBy(r => r, StringComparer.Ordinal))} ; "
                  + "fournie : allergenPropagation)");
foreach (var (name, r) in rules) {
    var prem = string.Join(" ; ", r.Premises.Select(p => $"({p[0]} {Court(p[1])} {p[2]})"));
    var concl = $"({r.Conclusion[0]} {Court(r.Conclusion[1])} {r.Conclusion[2]})";
    Console.WriteLine($"  {name,-20} {prem}  =>  {concl}");
}

Faits assertés   : 9


Règles           : 4  (intégrées : rdfs11, rdfs3, rdfs9 ; fournie : allergenPropagation)


  rdfs9                (?x type ?c) ; (?c subClassOf ?d)  =>  (?x type ?d)


  rdfs11               (?a subClassOf ?b) ; (?b subClassOf ?c)  =>  (?a subClassOf ?c)


  rdfs3                (?p range ?c) ; (?x ?p ?y)  =>  (?y type ?c)


  allergenPropagation  (?pizza containsIngredient ?ing) ; (?ing hasAllergen ?all)  =>  (?pizza hasAllergen ?all)


### Le moteur naïf : forward chaining qui enregistre le pourquoi

L'algorithme est volontairement naïf — ce n'est pas lui qui doit être subtil, c'est le certificat. À chaque tour, on essaie d'unifier les prémisses de chaque règle avec les triplets déjà connus ; chaque inférence **nouvelle** est ajoutée **et enregistrée comme une étape de dérivation** : la règle, la conclusion, les prémisses *concrètes* qui l'ont produite. On s'arrête au point fixe.

C'est la seule différence avec la clôture owlrl de SW-13 : owlrl rend un ensemble de triplets inférés ; ici, chaque triplet inféré arrive avec son *pourquoi*. Une variable liée deux fois à des valeurs différentes échoue (décision 0008 d'open-ontologies : *a binding is data and evidence admits one reading*).

In [3]:
// Moteur Horn naïf : forward chaining avec émission d'étapes de dérivation.
record EtapeDerivation(string Rule, (string S, string P, string O) Conclusion, List<(string S, string P, string O)> Premises);

static Dictionary<string, string>? MatchMoteur(string[] patron, (string S, string P, string O) triple) {
    // Unifie un triplet-patron avec un triplet ground ; renvoie {var: terme} ou null.
    var subst = new Dictionary<string, string>();
    var g = new[] { triple.S, triple.P, triple.O };
    for (int i = 0; i < 3; i++) {
        if (patron[i].StartsWith("?")) {
            if (subst.TryGetValue(patron[i], out var lie) && lie != g[i])
                return null;  // variable liée deux fois à des valeurs différentes
            subst[patron[i]] = g[i];
        } else if (patron[i] != g[i]) return null;
    }
    return subst;
}

static (string S, string P, string O) Instancier(string[] patron, Dictionary<string, string> subst)
    => (PatronTerm(patron[0], subst), PatronTerm(patron[1], subst), PatronTerm(patron[2], subst));
static string PatronTerm(string t, Dictionary<string, string> subst) => subst.TryGetValue(t, out var v) ? v : t;

static (List<(string S, string P, string O)> Known, List<EtapeDerivation> Derivations, int Iterations)
ForwardChain(List<(string S, string P, string O)> assertedIn,
             Dictionary<string, (string[][] Premises, string[] Conclusion)> rulesIn, int maxIter = 20) {
    // Forward chaining jusqu'au point fixe ; chaque inférence émet une étape.
    var known = new List<(string, string, string)>(assertedIn);
    var derivations = new List<EtapeDerivation>();
    bool changed = true; int iterations = 0;
    while (changed && iterations < maxIter) {
        changed = false; iterations++;
        foreach (var (name, rule) in rulesIn) {
            foreach (var triple in known.ToList()) {          // snapshot : prémisses 1
                var s1 = MatchMoteur(rule.Premises[0], triple);
                if (s1 == null) continue;
                if (rule.Premises.Length == 1) {
                    var concl = Instancier(rule.Conclusion, s1);
                    if (!known.Contains(concl)) {
                        known.Add(concl);
                        derivations.Add(new EtapeDerivation(name, concl, new() { triple }));
                        changed = true;
                    }
                    continue;
                }
                for (int j = 0; j < known.Count; j++) {        // liste vivante : prémisses 2
                    var t2 = known[j];
                    var patron2 = rule.Premises[1].Select(t => s1.TryGetValue(t, out var v) ? v : t).ToArray();
                    var s2 = MatchMoteur(patron2, t2);
                    if (s2 == null) continue;
                    var subst = new Dictionary<string, string>(s1);
                    foreach (var kv in s2) subst[kv.Key] = kv.Value;
                    var concl = Instancier(rule.Conclusion, subst);
                    if (!known.Contains(concl)) {
                        known.Add(concl);
                        derivations.Add(new EtapeDerivation(name, concl, new() { triple, t2 }));
                        changed = true;
                    }
                }
            }
        }
    }
    return (known, derivations, iterations);
}

var (known, derivations, iterations) = ForwardChain(asserted, rules);
Console.WriteLine($"Faits assertés   : {asserted.Count}");
Console.WriteLine($"Faits au point fixe : {known.Count}  ({derivations.Count} inférences, {iterations} itérations)");
Console.WriteLine("\nÉtapes de dérivation émises :");
for (int i = 0; i < derivations.Count; i++) {
    var d = derivations[i];
    var prem = string.Join(" ; ", d.Premises.Select(p => $"{Court(p.S)}/{Court(p.P)}/{Court(p.O)}"));
    Console.WriteLine($"  [{i}] {d.Rule,-20} {Court(d.Conclusion.S)}/{Court(d.Conclusion.P)}/{Court(d.Conclusion.O)}");
    Console.WriteLine($"       <- {prem}");
}

Faits assertés   : 9


Faits au point fixe : 18  (9 inférences, 3 itérations)



Étapes de dérivation émises :


  [0] rdfs9                margherita/type/Pizza


       <- margherita/type/NamedPizza ; NamedPizza/subClassOf/Pizza


  [1] rdfs9                mozzarella/type/AllergenSource


... (12 outputs left out)

       <- margherita/type/Pizza ; Pizza/subClassOf/Consumable


  [8] rdfs11               NamedPizza/subClassOf/Consumable


       <- NamedPizza/subClassOf/Pizza ; Pizza/subClassOf/Consumable



(4,34): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Interprétation — ce que le point fixe ne dit pas

**Sortie obtenue** : 9 faits assertés, 18 faits au point fixe, 9 inférences en 3 itérations.

| Aspect | Valeur | Signification |
|---|---|---|
| Inférences | 9 | le moteur produit du nouveau connaissable sans nous |
| Itérations | 3 | la chaîne `NamedPizza → Pizza → Food → Consumable` exige plusieurs tours |
| Étapes multi-sauts | dérivations 6, 7, 8 | leurs prémisses sont **elles-mêmes dérivées** (ex. `[7] margherita/type/Consumable` utilise `[0] margherita/type/Pizza` et `[3] Pizza/subClassOf/Consumable`) |
| Règle métier | `[5] margherita/hasAllergen/milk` | produite par `allergenPropagation`, la règle que *nous* avons écrite |

La conclusion « margherita est un allergène au lait » est vraie **pour ce moteur**. Mais la sortie ci-dessus est une liste imprimée : un tiers qui ne fait pas confiance au moteur ne peut rien en faire — il devrait ré-exécuter le même code, c'est-à-dire faire le même acte de confiance que celui qu'on voulait éviter. C'est précisément le problème que le certificat résout.

## 3. Émettre le certificat

Un **certificat d'inférence** est un fichier autonome contenant tout ce qu'un tiers doit connaître pour juger la conclusion — et rien d'autre :

1. la **table de règles** (patrons de prémisses/conclusion) ;
2. les **faits assertés** (le point de départ, tel que le certificat le déclare — voir la limite de provenance en section 7) ;
3. les **étapes de dérivation ordonnées** : règle, conclusion ground, prémisses ground ;
4. la **claim** à certifier.

Le format jouet est JSON (`sw16-toy-cert/1`), écrit par `System.Text.Json` — exactement le même schéma que le jumeau Python, pour que les deux certificats soient interopérables. open-ontologies utilise le même contenu en TSV (`oo-cert/1` : `règle <TAB> s p o <TAB> prémisses…`) — nous le vérifierons tel quel en section 6. La propriété décisive : **vérifier ne coûte pas plus que lire** — aucun raisonnement, aucune recherche de point fixe, juste une relecture d'étapes déjà écrites.

In [4]:
// Émission du certificat : le moteur sérialise règles + faits + dérivations.
var WORK = Directory.CreateDirectory("sw16-work");

var jsonOpts = new JsonSerializerOptions {
    WriteIndented = true,
    Encoder = System.Text.Encodings.Web.JavaScriptEncoder.UnsafeRelaxedJsonEscaping,
};
var compactOpts = new JsonSerializerOptions {
    Encoder = System.Text.Encodings.Web.JavaScriptEncoder.UnsafeRelaxedJsonEscaping,
};

static JsonArray VersTriplets(IEnumerable<(string S, string P, string O)> ts)
    => new JsonArray(ts.Select(t => (JsonNode?)new JsonArray(t.S, t.P, t.O)).ToArray());

var certificate = new JsonObject {
    ["format"] = "sw16-toy-cert/1",
    ["rule_table"] = new JsonObject(rules.Select(kv => new KeyValuePair<string, JsonNode?>(kv.Key,
        new JsonObject {
            ["premises"] = new JsonArray(kv.Value.Premises
                .Select(p => (JsonNode?)new JsonArray(p[0], p[1], p[2])).ToArray()),
            ["conclusion"] = new JsonArray(kv.Value.Conclusion[0], kv.Value.Conclusion[1], kv.Value.Conclusion[2]),
        }))),
    ["asserted"] = VersTriplets(asserted),
    ["derivations"] = new JsonArray(derivations.Select(d => (JsonNode?)new JsonObject {
        ["rule"] = d.Rule,
        ["conclusion"] = new JsonArray(d.Conclusion.S, d.Conclusion.P, d.Conclusion.O),
        ["premises"] = VersTriplets(d.Premises),
    }).ToArray()),
};

var certPath = Path.Combine(WORK.FullName, "certificat-margherita.json");
File.WriteAllText(certPath, certificate.ToJsonString(jsonOpts));
Console.WriteLine($"Certificat écrit : sw16-work/certificat-margherita.json  ({new FileInfo(certPath).Length} octets)");
Console.WriteLine($"  format      : {certificate["format"]}");
Console.WriteLine($"  règles      : {certificate["rule_table"]!.AsObject().Count}");
Console.WriteLine($"  assertés    : {certificate["asserted"]!.AsArray().Count}");
Console.WriteLine($"  dérivations : {certificate["derivations"]!.AsArray().Count}");

var der7 = certificate["derivations"]![7]!.AsObject();
var c7 = der7["conclusion"]!.AsArray();
Console.WriteLine($"\nExemple — étape 7 (multi-sauts, prémisses dérivées) :");
Console.WriteLine($"  règle      : {der7["rule"]}");
Console.WriteLine($"  conclusion : {Court(c7[0]!.GetValue<string>())}/{Court(c7[1]!.GetValue<string>())}/{Court(c7[2]!.GetValue<string>())}");
foreach (var p in der7["premises"]!.AsArray())
    Console.WriteLine($"  prémisses  : {Court(p![0]!.GetValue<string>())}/{Court(p![1]!.GetValue<string>())}/{Court(p![2]!.GetValue<string>())}");

Certificat écrit : sw16-work/certificat-margherita.json  (8630 octets)


  format      : sw16-toy-cert/1


  règles      : 4


  assertés    : 9


  dérivations : 9



Exemple — étape 7 (multi-sauts, prémisses dérivées) :


  règle      : rdfs9


  conclusion : margherita/type/Consumable


  prémisses  : margherita/type/Pizza


  prémisses  : Pizza/subClassOf/Consumable



(17,89): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(20,39): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(24,70): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(13,46): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Lecture du certificat

**Sortie obtenue** : le fichier `sw16-work/certificat-margherita.json` porte la table de règles, les 9 faits assertés et les 9 dérivations ordonnées.

L'étape 7 rend visible la chaîne multi-sauts : sa prémisse `margherita/type/Pizza` n'est **pas** un fait asserté — c'est la conclusion de l'étape 0. Le fichier est donc auto-porteur : qui le lit voit *où* chaque prémisse a été obtenue (assertée, ou dérivée plus haut), sans jamais rouvrir le moteur. C'est ce fichier — et lui seul — que la section suivante va juger.

## 4. Le checker indépendant

Le checker ne re-déduit **rien** : aucun point fixe, aucune recherche de règles applicables. Il relit chaque étape du certificat contre quatre exigences :

1. **Règle connue** — l'étape cite une règle de la table ;
2. **Une seule substitution** — les prémisses ground unifient avec les patrons **sous une même liaison des variables**, et la conclusion est *exactement* le patron de conclusion sous cette liaison (ground : plus aucune variable) ;
3. **Disponibilité ordonnée** — chaque prémisse est un fait asserté **ou** la conclusion d'une étape **strictement antérieure**. Interdit donc : l'auto-support (une étape qui se prend pour prémisse) et la prémisse du futur (une étape qui s'appuie sur ce qui sera prouvé plus bas) ;
4. **Claim atteinte** — la claim est la conclusion d'une étape acceptée.

Et l'indépendance n'est pas qu'une intention : le checker ci-dessous porte sa **propre unification** (`LierPatronChecker`), écrite sans appeler la moindre fonction du moteur — ni `MatchMoteur`, ni `ForwardChain`. La cellule le **prouve mécaniquement** en scannant l'IL compilé du checker : chaque instruction `call`/`callvirt` est résolue vers le nom de la méthode appelée, et aucun nom du moteur n'y figure. Un moteur bogué ne peut donc pas contaminer la vérification par un helper partagé.

Deux verdicts positifs, calqués sur la sémantique d'open-ontologies : `entailed` si seules des règles intégrées soutiennent la claim ; `entailed_under_supplied_rules` si une règle **fournie** intervient — une règle que vous avez écrite est une supposition que le certificat transporte, jamais un fait qu'il établit. Un verdict négatif est `rejected` avec des **raisons nommées**.

In [5]:
// Checker indépendant : vérifie le certificat, ne re-déduit rien.
public static class CheckerCertificat {

    public static Dictionary<string, string>? LierPatronChecker(string[] patron, (string S, string P, string O) ground) {
        // Unification propre au CHECKER — aucun appel au moteur (pas de MatchMoteur).
        // Renvoie {variable: terme} si le triplet ground incarne le patron, sinon null.
        var liaison = new Dictionary<string, string>();
        var g = new[] { ground.S, ground.P, ground.O };
        for (int i = 0; i < 3; i++) {
            if (patron[i].StartsWith("?")) {
                if (liaison.TryGetValue(patron[i], out var lie) && lie != g[i])
                    return null;  // variable liée deux fois à des valeurs différentes
                liaison[patron[i]] = g[i];
            } else if (patron[i] != g[i]) return null;  // terme constant : égalité stricte
        }
        return liaison;
    }

    static string T(string t, Dictionary<string, string> s) => s.TryGetValue(t, out var v) ? v : t;

    public static (string S, string P, string O) VersTuple(JsonArray a)
        => (a[0]!.GetValue<string>(), a[1]!.GetValue<string>(), a[2]!.GetValue<string>());

    static HashSet<string> RulesAtteintes(JsonObject cert, (string S, string P, string O) claim) {
        // Marche arrière : quelles règles soutiennent (transitivement) la claim.
        var derivationsArr = cert["derivations"]!.AsArray();
        var conclVersIdx = new Dictionary<(string, string, string), List<int>>();
        for (int i = 0; i < derivationsArr.Count; i++) {
            var c = VersTuple(derivationsArr[i]!["conclusion"]!.AsArray());
            if (!conclVersIdx.TryGetValue(c, out var l)) { l = new(); conclVersIdx[c] = l; }
            l.Add(i);
        }
        var used = new HashSet<string>();
        var stack = new Stack<(string, string, string)>();
        stack.Push((claim.S, claim.P, claim.O));
        var seen = new HashSet<(string, string, string)>();
        while (stack.Count > 0) {
            var t = stack.Pop();
            if (!seen.Add(t)) continue;
            foreach (var i in conclVersIdx.GetValueOrDefault(t, new List<int>())) {
                var d = derivationsArr[i]!.AsObject();
                used.Add(d["rule"]!.GetValue<string>());
                foreach (var p in d["premises"]!.AsArray())
                    stack.Push(VersTuple(p!.AsArray()));
            }
        }
        return used;
    }

    public static Dictionary<string, object?> VerifierCertificat(JsonObject cert,
            (string S, string P, string O) claim, HashSet<string> reglesIntegrees) {
        // Accepte ou rejette ; les rejets portent des raisons nommées.
        var problems = new List<(int Idx, string Rule, string Why)>();
        var rulesC = new Dictionary<string, (string[][] Premises, string[] Conclusion)>();
        foreach (var kv in cert["rule_table"]!.AsObject()) {
            var r = kv.Value!.AsObject();
            rulesC[kv.Key] = (
                r["premises"]!.AsArray().Select(p => p!.AsArray().Select(x => x!.GetValue<string>()).ToArray()).ToArray(),
                r["conclusion"]!.AsArray().Select(x => x!.GetValue<string>()).ToArray());
        }
        var available = new HashSet<(string, string, string)>(
            cert["asserted"]!.AsArray().Select(a => VersTuple(a!.AsArray())));
        var derivationsArr = cert["derivations"]!.AsArray();
        for (int i = 0; i < derivationsArr.Count; i++) {
            var d = derivationsArr[i]!.AsObject();
            var name = d["rule"]!.GetValue<string>();
            if (!rulesC.TryGetValue(name, out var rule)) { problems.Add((i, name, "UNKNOWN_RULE")); continue; }
            var premG = d["premises"]!.AsArray().Select(p => VersTuple(p!.AsArray())).ToList();
            var conclG = VersTuple(d["conclusion"]!.AsArray());
            if (premG.Count != rule.Premises.Length) { problems.Add((i, name, "MALFORMED_STEP")); continue; }
            var subst = new Dictionary<string, string>();
            bool okU = true;
            for (int k = 0; k < rule.Premises.Length; k++) {
                var s = LierPatronChecker(rule.Premises[k], premG[k]);
                if (s == null) { okU = false; problems.Add((i, name, "UNIFICATION_FAILED")); break; }
                foreach (var (nom, val) in s)
                    if (subst.TryGetValue(nom, out var ex) && ex != val)
                        { okU = false; problems.Add((i, name, "INCONSISTENT_BINDING")); break; }
                if (!okU) break;
                foreach (var (nom, val) in s) subst[nom] = val;
            }
            if (!okU) continue;
            var attendu = (T(rule.Conclusion[0], subst), T(rule.Conclusion[1], subst), T(rule.Conclusion[2], subst));
            if (conclG != attendu) { problems.Add((i, name, "CONCLUSION_MISMATCH")); continue; }
            if (conclG.S.StartsWith("?") || conclG.P.StartsWith("?") || conclG.O.StartsWith("?")) {
                problems.Add((i, name, "NON_GROUND_CONCLUSION")); continue;
            }
            foreach (var p in premG) {
                if (p == conclG) problems.Add((i, name, "SELF_SUPPORT"));
                else if (!available.Contains(p)) problems.Add((i, name, "PREMISE_NOT_AVAILABLE"));
            }
            // Une étape invalide ne contribue PAS sa conclusion : les étapes
            // suivantes ne peuvent pas s'appuyer sur un fait mal établi.
            if (!problems.Any(pr => pr.Idx == i)) available.Add(conclG);
        }
        var claimT = (claim.S, claim.P, claim.O);
        if (problems.Count > 0) {
            var uniques = problems.Distinct().ToList();
            return new Dictionary<string, object?> {
                ["ok"] = false, ["verdict"] = "rejected",
                ["problems"] = uniques.Select(pr => $"étape {pr.Idx} [{pr.Rule}] : {pr.Why}").ToList(),
                ["claim_supported"] = available.Contains(claimT),
            };
        }
        if (!available.Contains(claimT))
            return new Dictionary<string, object?> {
                ["ok"] = false, ["verdict"] = "rejected",
                ["problems"] = new List<string> { "CLAIM_NOT_DERIVED" }, ["claim_supported"] = false };
        var used = RulesAtteintes(cert, claimT);
        var verdict = used.IsSubsetOf(reglesIntegrees) ? "entailed" : "entailed_under_supplied_rules";
        return new Dictionary<string, object?> {
            ["ok"] = true, ["verdict"] = verdict, ["problems"] = new List<string>(),
            ["claim_supported"] = true, ["rules_used"] = used.OrderBy(r => r, StringComparer.Ordinal).ToList() };
    }
}

// Preuve mécanique du découplage : l'IL du checker ne référence AUCUNE
// méthode du moteur — ni l'unification MatchMoteur, ni ForwardChain().
static List<string> NomsMethodesAppelees(MethodInfo mi) {
    var noms = new List<string>();
    var il = mi.GetMethodBody()!.GetILAsByteArray();
    for (int i = 0; i + 5 <= il.Length; i++) {
        if (il[i] == 0x28 || il[i] == 0x6F) {  // call, callvirt
            int token = BitConverter.ToInt32(il, i + 1);
            try { noms.Add(mi.Module.ResolveMethod(token).Name); }
            catch { /* token non-méthode (champ, type, methodspec) : ignoré */ }
        }
    }
    return noms;
}

var appelsChecker = NomsMethodesAppelees(typeof(CheckerCertificat).GetMethod("VerifierCertificat")!)
    .Distinct().OrderBy(n => n).ToList();
if (appelsChecker.Any(n => n is "MatchMoteur" or "ForwardChain"))
    throw new InvalidOperationException("checker -> moteur !");
Console.WriteLine("Découplage vérifié : VerifierCertificat n'appelle aucune méthode du moteur "
    + "(unification propre : LierPatronChecker).");
Console.WriteLine("  méthodes appelées par son IL : " + string.Join(", ", appelsChecker));

var claimAllergene = (Q("margherita"), Q("hasAllergen"), Q("milk"));
var claimType = (Q("margherita"), RdfType, Q("Consumable"));
Console.WriteLine("Claim 1 — margherita/hasAllergen/milk (règle métier impliquée) :");
Console.WriteLine("  " + JsonSerializer.Serialize(
    CheckerCertificat.VerifierCertificat(certificate, claimAllergene, builtinRules), compactOpts));
Console.WriteLine("Claim 2 — margherita/type/Consumable (règles intégrées seules) :");
Console.WriteLine("  " + JsonSerializer.Serialize(
    CheckerCertificat.VerifierCertificat(certificate, claimType, builtinRules), compactOpts));

Découplage vérifié : VerifierCertificat n'appelle aucune méthode du moteur (unification propre : LierPatronChecker).


  méthodes appelées par son IL : .ctor, Add, Any, AsArray, AsObject, Contains, Deconstruct, Dispose, Distinct, get_Count, get_Current, get_Item, get_Key, get_Ordinal, get_Value, GetEnumerator, GetValue, IsSubsetOf, LierPatronChecker, MoveNext, op_Equality, op_Inequality, OrderBy, RulesAtteintes, Select, set_Item, StartsWith, T, ToArray, ToList, TryGetValue, VersTuple


Claim 1 — margherita/hasAllergen/milk (règle métier impliquée) :


  {"ok":true,"verdict":"entailed_under_supplied_rules","problems":[],"claim_supported":true,"rules_used":["allergenPropagation"]}


Claim 2 — margherita/type/Consumable (règles intégrées seules) :


  {"ok":true,"verdict":"entailed","problems":[],"claim_supported":true,"rules_used":["rdfs11","rdfs9"]}



(4,45): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(50,44): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(99,49): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(106,49): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(111,45): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Interprétation — deux verdicts, pas un

**Sortie obtenue** : la claim allergène reçoit `entailed_under_supplied_rules` avec `rules_used: ["allergenPropagation"]` ; la claim de type reçoit `entailed` avec `rules_used: ["rdfs11", "rdfs9"]`.

| Claim | Verdict | Pourquoi |
|---|---|---|
| `margherita/hasAllergen/milk` | `entailed_under_supplied_rules` | la marche arrière ne trouve que `allergenPropagation` — une règle **fournie** |
| `margherita/type/Consumable` | `entailed` | uniquement des règles intégrées (`rdfs9`, `rdfs11`) — vrai dans tout modèle du graphe asserté |

Cette distinction est une **fonctionnalité d'honnêteté**, pas un détail : elle empêche la règle métier de se déguiser en fait établi. open-ontologies pousse la même exigence jusqu'au test — un test échoue si le verdict cesse de changer quand une règle fournie entre en jeu. Le fil rouge s'enrichit : *sans preuve*, ces deux claims auraient le même statut (« le raisonneur l'a dit »).

## 5. Tests négatifs — quatre certificats forgés, quatre rejets nommés

Un checker qui accepterait tout serait décoratif : la valeur se **démontre** en essayant de le tromper. Chaque forge ci-dessous est une attaque nommée contre une exigence du checker :

| Forge | Attaque | Exigence visée |
|---|---|---|
| **A — allergène fabriqué** | ajoute « margherita porte le gluten » avec une prémisse inventée (`mozzarella/hasAllergen/gluten`, jamais assertée ni dérivée) | disponibilité des prémisses |
| **B — auto-support** | « la classe Spicy est sous-classe d'elle-même » en se prenant elle-même pour prémisse | pas de circularité |
| **C — règle trompeuse** | une dérivation étiquetée `rdfs9` dont les prémisses sont celles de `allergenPropagation` | une seule substitution |
| **D — prémisses du futur** | la chaîne multi-sauts est réordonnée : l'étape 7 placée **avant** l'étape 0 dont elle dépend | disponibilité **ordonnée** |

La forge D reproduit en miniature la classe de défauts que le différentiel multi-kernels d'open-ontologies a réellement attrapés sur son propre moteur (README v1.4.0 : cinq « false cleans », une règle concluant un triplet non sérialisable, deux kernels en désaccord sur des certificats pourtant valides de part et d'autre).

In [6]:
// Quatre forges, quatre rejets nommés — le résultat falsifiable.
static JsonObject CopieProfonde(JsonObject o) => (JsonObject)JsonNode.Parse(o.ToJsonString())!;

static JsonObject ForgeAjout(JsonObject cert, JsonObject etape) {
    var f = CopieProfonde(cert);
    f["derivations"]!.AsArray().Add(CopieProfonde(etape));
    return f;
}

static JsonObject Etape(string rule, (string S, string P, string O) concl,
                        params (string S, string P, string O)[] prems)
    => new JsonObject {
        ["rule"] = rule,
        ["conclusion"] = new JsonArray(concl.S, concl.P, concl.O),
        ["premises"] = new JsonArray(prems.Select(p => (JsonNode?)new JsonArray(p.S, p.P, p.O)).ToArray()),
    };

// NB : chaque étape passe par une variable — un appel `params` expansé
// imbriqué dans un autre appel (ForgeAjout(cert, Etape(...))) ne compile
// pas sous le parseur C# scripting (CS1026).
var etapeA = Etape("allergenPropagation",
    (Q("margherita"), Q("hasAllergen"), Q("gluten")),
    (Q("margherita"), Q("containsIngredient"), Q("mozzarella")),
    (Q("mozzarella"), Q("hasAllergen"), Q("gluten")));
var forgeA = ForgeAjout(certificate, etapeA);

var etapeB = Etape("rdfs11",
    (Q("Spicy"), RdfsSubClass, Q("Spicy")),
    (Q("Spicy"), RdfsSubClass, Q("Spicy")), (Q("Spicy"), RdfsSubClass, Q("Spicy")));
var forgeB = ForgeAjout(certificate, etapeB);

var etapeC = Etape("rdfs9",
    (Q("margherita"), RdfType, Q("AllergenSource")),
    (Q("margherita"), Q("containsIngredient"), Q("mozzarella")),
    (Q("mozzarella"), Q("hasAllergen"), Q("milk")));
var forgeC = ForgeAjout(certificate, etapeC);

var forgeD = CopieProfonde(certificate);
{
    var d = forgeD["derivations"]!.AsArray();
    // Swap par copies détachées : un JsonNode déjà parenté ne se réinsère pas
    // tel quel (System.Text.Json refuse le double parentage).
    var e0 = (JsonNode?)d[0]!.DeepClone();
    var e7 = (JsonNode?)d[7]!.DeepClone();
    d[0] = e7;  // l'étape 7 passe AVANT l'étape 0 dont elle utilise la conclusion
    d[7] = e0;
}

foreach (var (nom, forge, claim) in new[] {
    ("Forge A — allergène fabriqué", forgeA, (Q("margherita"), Q("hasAllergen"), Q("gluten"))),
    ("Forge B — auto-support",       forgeB, claimAllergene),
    ("Forge C — règle trompeuse",    forgeC, claimAllergene),
    ("Forge D — prémisses du futur", forgeD, claimAllergene),
}) {
    var verdict = CheckerCertificat.VerifierCertificat(forge, claim, builtinRules);
    Console.WriteLine($"{nom} : verdict = {((string)verdict["verdict"]!).ToUpper()}");
    foreach (var p in (List<string>)verdict["problems"]!)
        Console.WriteLine($"    REJET — {p}");
}

Forge A — allergène fabriqué : verdict = REJECTED


    REJET — étape 9 [allergenPropagation] : PREMISE_NOT_AVAILABLE


Forge B — auto-support : verdict = REJECTED


    REJET — étape 9 [rdfs11] : SELF_SUPPORT


Forge C — règle trompeuse : verdict = REJECTED


    REJET — étape 9 [rdfs9] : UNIFICATION_FAILED


Forge D — prémisses du futur : verdict = REJECTED


    REJET — étape 0 [rdfs9] : PREMISE_NOT_AVAILABLE



(43,23): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(44,23): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(15,65): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Interprétation — ce que les rejets garantissent

**Sortie obtenue** : les quatre forges sont rejetées, chacune pour une raison nommée — `PREMISE_NOT_AVAILABLE` (A), `SELF_SUPPORT` (B), `UNIFICATION_FAILED` (C), `PREMISE_NOT_AVAILABLE` sur l'étape 0 réordonnée (D).

| Forge | Raison du rejet | Ce que le checker vient de prouver |
|---|---|---|
| A | `PREMISE_NOT_AVAILABLE` | on ne fabrique pas un fait en inventant une prémisse |
| B | `SELF_SUPPORT` | une preuve ne peut pas se prendre elle-même pour point de départ |
| C | `UNIFICATION_FAILED` | coller l'étiquette d'une règle intégrée sur des prémisses étrangères ne passe pas |
| D | `PREMISE_NOT_AVAILABLE` (étape 0) | l'**ordre** du certificat porte du sens : une étape ne peut s'appuyer que sur ce qui est déjà établi |

Deux remarques d'honnêteté. D'abord, ces forges attaquent le **certificat** ; le moteur, lui, reste non fiable par hypothèse — c'est le régime de proof-carrying : *l'incrédulité envers le moteur est le point de départ, pas un accident*. Ensuite, notre checker jouet partage le fichier avec le moteur dans ce notebook ; l'indépendance **réelle** — deux langages, deux historiques de code, un théorème de correction — est celle du checker Lean face au moteur Rust, que la section 6 cite à titre de jambe réelle.

## Exemple guidé 1 — Étendre l'ontologie et certifier une nouvelle claim

**Situation** : la pizzeria ajoute la sauce tomate à sa margherita, et la sauce tomate porte l'allergène *nightshade* (famille des solanacées). Nous déroulons le cycle complet **ensemble** — l'exercice 1 ci-dessous interrogera le checker sous un angle que cet exemple ne couvre pas : les claims absentes.

(a) Étendre `asserted` avec quatre faits : `margherita containsIngredient tomatoSauce`, `tomatoSauce hasAllergen nightshade`, `tomatoSauce type VegetableProduct` et `VegetableProduct subClassOf AllergenSource` ;
(b) relancer le moteur (`ForwardChain`) et **émettre** le certificat étendu ;
(c) vérifier avec `VerifierCertificat` la claim `margherita/hasAllergen/nightshade` — quel verdict, et pourquoi ;
(d) vérifier la claim `tomatoSauce/type/AllergenSource` — quel verdict, et quelle différence avec (c) ?

Indice de lecture : le verdict dépend de la marche arrière `RulesAtteintes` — quelles règles soutiennent chaque claim ?

In [7]:
// Exemple guidé 1 — ontologie étendue, certificat émis, deux verdicts.
var assertedEx1 = new List<(string, string, string)>(asserted) {
    (Q("margherita"), Q("containsIngredient"), Q("tomatoSauce")),
    (Q("tomatoSauce"), Q("hasAllergen"), Q("nightshade")),
    (Q("tomatoSauce"), RdfType, Q("VegetableProduct")),
    (Q("VegetableProduct"), RdfsSubClass, Q("AllergenSource")),
};
var (knownEx1, derivationsEx1, itersEx1) = ForwardChain(assertedEx1, rules);
var certificatEx1 = new JsonObject {
    ["format"] = "sw16-toy-cert/1",
    ["rule_table"] = certificate["rule_table"]!.DeepClone()!,   // même table, recopiée
    ["asserted"] = new JsonArray(assertedEx1
        .Select(t => (JsonNode?)new JsonArray(t.Item1, t.Item2, t.Item3)).ToArray()),
    ["derivations"] = new JsonArray(derivationsEx1.Select(d => (JsonNode?)new JsonObject {
        ["rule"] = d.Rule,
        ["conclusion"] = new JsonArray(d.Conclusion.S, d.Conclusion.P, d.Conclusion.O),
        ["premises"] = new JsonArray(d.Premises
            .Select(p => (JsonNode?)new JsonArray(p.S, p.P, p.O)).ToArray()),
    }).ToArray()),
};
Console.WriteLine($"Ontologie étendue : {assertedEx1.Count} assertés, "
    + $"{derivationsEx1.Count} dérivations en {itersEx1} itérations");

var claimNightshade = (Q("margherita"), Q("hasAllergen"), Q("nightshade"));
var claimVeg = (Q("tomatoSauce"), RdfType, Q("AllergenSource"));
var v1 = CheckerCertificat.VerifierCertificat(certificatEx1, claimNightshade, builtinRules);
var v2 = CheckerCertificat.VerifierCertificat(certificatEx1, claimVeg, builtinRules);
Console.WriteLine("Claim (c) margherita/hasAllergen/nightshade : "
    + JsonSerializer.Serialize(v1, compactOpts));
Console.WriteLine("Claim (d) tomatoSauce/type/AllergenSource   : "
    + JsonSerializer.Serialize(v2, compactOpts));

Ontologie étendue : 13 assertés, 12 dérivations en 3 itérations


Claim (c) margherita/hasAllergen/nightshade : {"ok":true,"verdict":"entailed_under_supplied_rules","problems":[],"claim_supported":true,"rules_used":["allergenPropagation"]}


Claim (d) tomatoSauce/type/AllergenSource   : {"ok":true,"verdict":"entailed","problems":[],"claim_supported":true,"rules_used":["rdfs9"]}



(13,31): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(18,35): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(14,73): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Lecture — Exemple guidé 1 : deux statuts épistémiques dans un même fichier

La claim (c) reçoit `entailed_under_supplied_rules` — elle repose sur `allergenPropagation`, une règle fournie. La claim (d) reçoit `entailed` — elle ne mobilise que `rdfs9` sur des faits assertés. Le même certificat porte donc **deux statuts épistémiques différents**, lisibles dans le fichier : c'est toute la différence avec SW-13, où les deux conclusions seraient de simples triplets inférés, indiscernables.

## Exercice 1 — Une claim absente : que répond le checker, exactement ?

**Contexte** : l'exemple guidé 1 n'a interrogé le checker que sur des claims **soutenues**. Un checker honnête doit aussi répondre — correctement — quand on lui demande une claim qui n'est **pas** dans le certificat, ou une claim qui n'y figure que comme fait **asserté**, jamais dérivée.

**Objectif** : sur le certificat honnête `certificate` de la section 3, **sans rien lui ajouter**, exécuter `VerifierCertificat` sur deux claims et interpréter la différence :

(a) la claim `(margherita, hasAllergen, gluten)` — le gluten ne figure nulle part dans l'ontologie. Verdict, `problems`, `claim_supported` ? Notez la forme de ce rejet : c'est le seul qui ne cite **aucune étape** ;
(b) la claim `(mozzarella, hasAllergen, milk)` — un fait asserté, jamais dérivé. Verdict, et que vaut `rules_used` ?
(c) en une phrase : pourquoi le checker répond-il `rejected` plutôt que « cette claim est fausse » — qu'est-ce que le checker ne sait tout simplement pas dire ?

> **Indice :** suivez le code de `VerifierCertificat` APRÈS la boucle sur les étapes — que se passe-t-il quand `problems` est vide mais que la claim n'est pas dans `available` ? Et pour (b), que renvoie `RulesAtteintes` quand aucune dérivation ne conclut le triplet ?
- **Etape 1 :** `v1 = CheckerCertificat.VerifierCertificat(certificate, (Q("margherita"), Q("hasAllergen"), Q("gluten")), builtinRules)`
- **Etape 2 :** `v2 = CheckerCertificat.VerifierCertificat(certificate, (Q("mozzarella"), Q("hasAllergen"), Q("milk")), builtinRules)`
- **Etape 3 :** lire verdict / problems / rules_used de chacun, puis formuler (c)

In [8]:
// Exercice 1 — à compléter
// Etape 1 : la claim absente (gluten) sur le certificat honnête
// Etape 2 : la claim assertée jamais dérivée (mozzarella/lait)
// Etape 3 : lecture des verdicts et réponse à (c)
object resultatEx1 = null;  // TODO étudiant : (verdict gluten, verdict mozzarella, réponse à (c))
Console.WriteLine("Exercice 1 à compléter");

Exercice 1 à compléter


## 6. La jambe réelle — le binaire open-ontologies v1.4.0

Le jouet démontre le *concept* ; la comparaison exige le *vrai outil*. **open-ontologies** (Fabio Rovai, MIT) est une plateforme d'ingénierie et de vérification d'ontologies : moteur de raisonnement Rust, checkers Lean 4 dont la correction est machine-checkée, second kernel Isabelle, studio web — et un préprint ([arXiv:2605.09184](https://arxiv.org/abs/2605.09184)). Le dépôt évolue vite : toute utilisation **fige le tag**. Ici :

| Élément | Valeur figée |
|---|---|
| Version | **v1.4.0** (release publiée le 2026-09-15) |
| Binaire Windows | `open-ontologies-x86_64-pc-windows-msvc.exe` |
| SHA256 (SHASUMS.txt officiel) | `e1e1334408227d208837de1b389b6b9aad24cc87b825de8c28480f2c39b210d5` |
| Outillage annoncé au tag | 114 outils (le compte dérive vite — section 8) |

**Résolution du binaire** (dans l'ordre) : variable d'environnement `OPEN_ONTOLOGIES_BIN` → `ext_tools/` local à la série → répertoire courant → **téléchargement automatique** depuis l'URL épinglée de la release (58 Mo, une seule fois, puis cache). Le SHA256 est recontrôlé à chaque résolution : un binaire dont l'empreinte ne correspond pas au pin n'est **pas** invoqué. C'est la règle du dépôt : **installer, jamais contourner** — et ici l'installation est le geste du carnet lui-même (`HttpClient`, puis `System.Diagnostics.Process` pour l'invoquer).

In [9]:
// Résolution (ou installation) du binaire épinglé v1.4.0 — règle : installer, jamais contourner.
var pinnedVersion = "v1.4.0";
var pinnedSha256 = new Dictionary<string, string> {
    ["open-ontologies-x86_64-pc-windows-msvc.exe"] = "e1e1334408227d208837de1b389b6b9aad24cc87b825de8c28480f2c39b210d5",
    ["open-ontologies-x86_64-unknown-linux-gnu"]   = "8cdc57af888a9a0578ccb3646dfc28757cbb533d155f368788b7df5543a7ef56",
    ["open-ontologies-aarch64-apple-darwin"]       = "85db2eadd74331eaca51fd8b535e51ea8ecb9dfb396cae46fdc087ba3c84ee63",
    ["open-ontologies-x86_64-apple-darwin"]        = "ddbdb980c24a57a93c65edf08d3bfa515843148f61e7a9bc8b6c8aeda7afa55c",
};
string asset =
    RuntimeInformation.IsOSPlatform(OSPlatform.Windows) ? "open-ontologies-x86_64-pc-windows-msvc.exe"
    : RuntimeInformation.IsOSPlatform(OSPlatform.OSX)
        ? (RuntimeInformation.ProcessArchitecture == Architecture.Arm64
               ? "open-ontologies-aarch64-apple-darwin" : "open-ontologies-x86_64-apple-darwin")
    : RuntimeInformation.IsOSPlatform(OSPlatform.Linux) ? "open-ontologies-x86_64-unknown-linux-gnu"
    : throw new NotSupportedException($"plateforme non supportée : {RuntimeInformation.OSDescription}");
var pinnedUrl = $"https://github.com/fabio-rovai/open-ontologies/releases/download/{pinnedVersion}/{asset}";

static string Sha256Fichier(string path)
    => Convert.ToHexString(SHA256.HashData(File.ReadAllBytes(path))).ToLowerInvariant();

string ResoudreBinaire() {
    // env OPEN_ONTOLOGIES_BIN -> ext_tools/ -> répertoire courant -> téléchargement épinglé.
    var env = Environment.GetEnvironmentVariable("OPEN_ONTOLOGIES_BIN");
    if (env != null && File.Exists(env)) return env;
    var local = Path.Combine("ext_tools", asset);
    if (File.Exists(local)) return local;
    foreach (var cand in new[] { asset, Path.ChangeExtension(asset, null) })
        if (File.Exists(cand)) return cand;
    Console.WriteLine($"Binaire absent — téléchargement épinglé {pinnedVersion} (~58 Mo, une fois)…");
    Directory.CreateDirectory("ext_tools");
    using var hc = new HttpClient();
    File.WriteAllBytes(local, hc.GetByteArrayAsync(pinnedUrl).Result);
    return local;
}

var ooBin = ResoudreBinaire();
var empreinte = Sha256Fichier(ooBin);
var attendue = pinnedSha256[asset];
bool integriteOk = empreinte == attendue;
Console.WriteLine($"Binaire     : {ooBin}  ({new FileInfo(ooBin).Length:N0} octets)");
Console.WriteLine($"SHA256      : {empreinte}");
Console.WriteLine($"Attendu     : {attendue}");
Console.WriteLine($"Intégrité   : {(integriteOk ? "OK — empreinte conforme au pin v1.4.0" : "ÉCHEC — binaire non invoqué")}");

Binaire     : ext_tools\open-ontologies-x86_64-pc-windows-msvc.exe  (60 875 264 octets)


SHA256      : e1e1334408227d208837de1b389b6b9aad24cc87b825de8c28480f2c39b210d5


Attendu     : e1e1334408227d208837de1b389b6b9aad24cc87b825de8c28480f2c39b210d5


Intégrité   : OK — empreinte conforme au pin v1.4.0


### Charger la vraie pizza et raisonner avec certificat

La fixture est `data/pizza.owl` — l'ontologie pizza canonique (co-ode) déjà livrée dans la série pour SW-7 et SW-13, donc une **vraie ontologie, déjà sourcée**, pas un exemple inventé. Le binaire fonctionne en *batch* : `load` puis `reason --profile rdfs --certificate <dir>` — le store étant en mémoire, les deux commandes doivent partager un même batch. Côté C#, le binaire est invoqué par `System.Diagnostics.Process` (flux stdout redirigé, encodage UTF-8), et chaque ligne JSON de la réponse est relue par `System.Text.Json`. Le répertoire cible reçoit le certificat `oo-cert/1` (`asserted.tsv` + `derivations.tsv`) que le champ `check_with` invite à faire vérifier par le checker Lean (`lake exe oo-cert …`).

In [10]:
// Jambe réelle : batch load + reason --certificate sur data/pizza.owl.
var ooData = Path.Combine(WORK.FullName, "oo-data");
var certPizza = Path.Combine(WORK.FullName, "cert-pizza");
if (!integriteOk) {
    Console.WriteLine("ÉCHEC d'intégrité du binaire — invocation refusée (pin SHA256 non conforme).");
} else {
    if (Directory.Exists(certPizza)) Directory.Delete(certPizza, recursive: true);
    var batch = Path.Combine(WORK.FullName, "batch-pizza.txt");
    File.WriteAllText(batch,
        "load data/pizza.owl\n" +
        "stats\n" +
        $"reason --profile rdfs --certificate {certPizza.Replace('\\', '/')}\n",
        new UTF8Encoding(false));
    var psi = new ProcessStartInfo {
        FileName = Path.GetFullPath(ooBin),
        Arguments = $"batch \"{Path.GetFullPath(batch)}\" --data-dir \"{Path.GetFullPath(ooData)}\" --no-connect",
        WorkingDirectory = Directory.GetCurrentDirectory(),   // data/pizza.owl relatif au dossier de série
        RedirectStandardOutput = true, RedirectStandardError = true,
        StandardOutputEncoding = Encoding.UTF8, StandardErrorEncoding = Encoding.UTF8,
        UseShellExecute = false, CreateNoWindow = true,
    };
    using var proc = Process.Start(psi)!;
    var stdoutTask = proc.StandardOutput.ReadToEndAsync();
    var stderrTask = proc.StandardError.ReadToEndAsync();
    if (!proc.WaitForExit(300_000)) { proc.Kill(); throw new TimeoutException("batch open-ontologies > 300 s"); }
    var reponses = stdoutTask.Result.Split('\n')
        .Where(l => l.TrimStart().StartsWith("{"))
        .Select(l => JsonNode.Parse(l.Trim())!.AsObject())
        .ToList();
    foreach (var r in reponses) {
        var cmd = r["command"]!.GetValue<string>();
        if (cmd == "load")
            Console.WriteLine($"load  : {r["result"]!["triples_loaded"]} triplets chargés");
        else if (cmd == "stats") {
            var s = r["result"]!;
            Console.WriteLine($"stats : {s["triples"]} triplets, {s["classes"]} classes, "
                + $"{s["individuals"]} individus, {s["object_properties"]} propriétés d'objet");
        }
    }
    var raison = reponses.First(r => r["command"]!.GetValue<string>() == "reason")["result"]!.AsObject();
    var cert = raison["certificate"]!.AsObject();
    Console.WriteLine($"reason: {raison["inferred_count"]} inférences en {raison["iterations"]} itérations "
        + $"(point fixe : {raison["fixpoint_reached"]})");
    Console.WriteLine($"  par règle    : {cert["by_rule"]}");
    Console.WriteLine($"  format       : {cert["format"]}");
    Console.WriteLine($"  dérivations  : {cert["derivations"]}  |  assertés : {cert["asserted"]}");
    Console.WriteLine($"  vérif. Lean  : {cert["check_with"]}");
}

load  : 1944 triplets chargés


stats : 1944 triplets, 99 classes, 218 individus, 8 propriétés d'objet


reason: 258 inférences en 4 itérations (point fixe : true)


  par règle    : {
  "rdfs11": 258
}


  format       : oo-cert/1


  dérivations  : 258  |  assertés : 1944


  vérif. Lean  : cd lean && lake exe oo-cert <dir>/asserted.tsv <dir>/derivations.tsv


### Interprétation — la jambe réelle tourne

**Sortie obtenue** : la fixture charge 1944 triplets (99 classes, 218 individus) ; le profil `rdfs` dérive **258 inférences en 4 itérations**, toutes par la règle `rdfs11` (transitivité de `subClassOf`) ; le certificat `oo-cert/1` est écrit avec 258 dérivations.

| Aspect | Valeur mesurée | Signification |
|---|---|---|
| Triplets assertés | 1944 | vraie ontologie, déjà utilisée par SW-7/SW-13 |
| Inférences | 258 | la hiérarchie de classes de la pizza est profonde — la transitivité travaille |
| Règles mobilisées | `rdfs11` seule | sur cette ontologie, le profil RDFS ne tire que de la transitivité |
| Certificat | `oo-cert/1`, 258 étapes | le moteur **émet** les prémisses ground de chaque dérivation — même format d'idée que notre jouet |
| Vérification annoncée | `lake exe oo-cert …` | la jambe *indépendante* est le checker Lean (section 7 pour ses limites) |

Note d'honnêteté (README v1.4.0) : *« The proofs are ahead of the published release »* — la couche vérifiée décrite dans le README vit sur `main`, plus récente que le tag ; le binaire de release embarque le **moteur** et l'**émission** de certificats, la preuve de correction du checker se lit dans `lean/OOCert` au tag. C'est exactement le genre de précision que la section 8 érige en discipline.

### Notre checker jouet vérifie le certificat réel

Le format `oo-cert/1` est un TSV : `règle <TAB> s p o <TAB> s p o [<TAB> s p o]` — règle, conclusion, puis une prémisse par groupe de trois (les mêmes informations que notre JSON jouet, sérialisées autrement). Notre `VerifierOoCert` ci-dessous applique la **même procédure** (unification sous une seule substitution, disponibilité ordonnée) avec la **même unification propre au checker** (`LierPatronChecker`) — aucune fonction du moteur n'intervient ici non plus. Deux exécutions : le certificat **tel qu'émis**, puis une copie **trafiquée** (le sujet de la première conclusion est réécrit) — un checker qui n'accepterait que ne vaudrait rien.

In [11]:
// Vérification du certificat oo-cert/1 réel — puis d'une copie trafiquée.
public static class CheckerTsv {

    static (string, string, string) VersTermsTsv(string line) {
        var c = line.TrimEnd('\n', '\r').Split('\t');
        return (c[0].Trim().Trim('<', '>'), c[1].Trim().Trim('<', '>'), c[2].Trim().Trim('<', '>'));
    }

    public static Dictionary<string, object?> VerifierOoCert(string assertedPath, string derivationsPath,
            Dictionary<string, (string[][] Premises, string[] Conclusion)> ruleTable) {
        // Même procédure que VerifierCertificat, sur le TSV oo-cert/1 —
        // et même unification propre au checker (LierPatronChecker, pas MatchMoteur).
        var problems = new List<(int Idx, string Rule, string Why)>();
        var available = new HashSet<(string, string, string)>();
        foreach (var line in File.ReadLines(assertedPath, Encoding.UTF8)) {
            if (string.IsNullOrWhiteSpace(line)) continue;
            available.Add(VersTermsTsv(line));
        }
        int n = 0; int i = -1;
        foreach (var line in File.ReadLines(derivationsPath, Encoding.UTF8)) {
            i++;
            if (string.IsNullOrWhiteSpace(line)) continue;
            n++;
            var fields = line.TrimEnd('\n', '\r').Split('\t');
            var name = fields[0].Trim();
            var terms = fields.Skip(1).Select(c => c.Trim().Trim('<', '>')).ToArray();
            if (!ruleTable.TryGetValue(name, out var rule)) { problems.Add((i, name, "UNKNOWN_RULE")); continue; }
            int nprem = rule.Premises.Length;
            if (terms.Length != 3 * (1 + nprem)) { problems.Add((i, name, "ARITY")); continue; }
            var conclG = (terms[0], terms[1], terms[2]);
            var premG = Enumerable.Range(0, nprem)
                .Select(k => (terms[3 + 3 * k], terms[4 + 3 * k], terms[5 + 3 * k])).ToList();
            var subst = new Dictionary<string, string>();
            bool okU = true;
            for (int k = 0; k < nprem; k++) {
                var s = CheckerCertificat.LierPatronChecker(rule.Premises[k], premG[k]);
                if (s == null) { okU = false; problems.Add((i, name, "UNIFICATION_FAILED")); break; }
                foreach (var (nom, val) in s)
                    if (subst.TryGetValue(nom, out var ex) && ex != val)
                        { okU = false; break; }
                if (!okU) break;
                foreach (var (nom, val) in s) subst[nom] = val;
            }
            if (!okU) { problems.Add((i, name, "INCONSISTENT_BINDING")); continue; }
            var attendu = (rule.Conclusion[0].StartsWith("?") ? subst.GetValueOrDefault(rule.Conclusion[0]) : rule.Conclusion[0],
                          rule.Conclusion[1].StartsWith("?") ? subst.GetValueOrDefault(rule.Conclusion[1]) : rule.Conclusion[1],
                          rule.Conclusion[2].StartsWith("?") ? subst.GetValueOrDefault(rule.Conclusion[2]) : rule.Conclusion[2]);
            if (conclG != attendu) { problems.Add((i, name, "CONCLUSION_MISMATCH")); continue; }
            foreach (var p in premG) {
                if (p == conclG) problems.Add((i, name, "SELF_SUPPORT"));
                else if (!available.Contains(p)) problems.Add((i, name, "PREMISE_NOT_AVAILABLE"));
            }
            // Une étape invalide ne contribue PAS sa conclusion.
            if (!problems.Any(pr => pr.Idx == i)) available.Add(conclG);
        }
        return new Dictionary<string, object?> {
            ["steps"] = n,
            ["total_problems"] = problems.Count,
            ["reasons"] = problems.Select(p => p.Why).Distinct().OrderBy(x => x).ToList(),
            ["problems"] = problems.Distinct().Take(3).Select(pr => $"étape {pr.Idx} [{pr.Rule}] : {pr.Why}").ToList(),
        };
    }
}

// Le checker TSV non plus ne référence aucune fonction du moteur.
var ilTsv = NomsMethodesAppelees(typeof(CheckerTsv).GetMethod("VerifierOoCert")!)
    .Distinct().OrderBy(x => x).ToList();
if (ilTsv.Any(nm => nm is "MatchMoteur" or "ForwardChain"))
    throw new InvalidOperationException("checker TSV -> moteur !");
Console.WriteLine("Découplage vérifié : VerifierOoCert utilise LierPatronChecker, pas MatchMoteur().");

var ooRules = new Dictionary<string, (string[][] Premises, string[] Conclusion)> {
    ["rdfs9"] = (new[] { new[] { "?x", RdfType, "?c" }, new[] { "?c", RdfsSubClass, "?d" } },
                 new[] { "?x", RdfType, "?d" }),
    ["rdfs11"] = (new[] { new[] { "?a", RdfsSubClass, "?b" }, new[] { "?b", RdfsSubClass, "?c" } },
                  new[] { "?a", RdfsSubClass, "?c" }),
};

if (integriteOk) {
    var okCert = CheckerTsv.VerifierOoCert(Path.Combine(certPizza, "asserted.tsv"),
                                           Path.Combine(certPizza, "derivations.tsv"), ooRules);
    Console.WriteLine("Certificat réel tel qu'émis      : " + JsonSerializer.Serialize(okCert, compactOpts));

    // Forge E : copie trafiquée — le sujet de la conclusion de l'étape 0 est réécrit
    // (suffixe FORGE inséré dans l'IRI, indépendant du contenu de la ligne).
    var trafiqueDir = Path.Combine(WORK.FullName, "cert-pizza-trafique");
    if (Directory.Exists(trafiqueDir)) Directory.Delete(trafiqueDir, recursive: true);
    Directory.CreateDirectory(trafiqueDir);
    File.Copy(Path.Combine(certPizza, "asserted.tsv"), Path.Combine(trafiqueDir, "asserted.tsv"));
    var lignes = File.ReadAllLines(Path.Combine(certPizza, "derivations.tsv"), Encoding.UTF8).ToList();
    var champs = lignes[0].Split('\t').ToList();
    var sujet = champs[1];
    if (!(sujet.StartsWith("<") && sujet.EndsWith(">"))) throw new InvalidOperationException(sujet);
    champs[1] = sujet[..^1] + "FORGE>";  // <...quelquechoseFORGE>
    lignes[0] = string.Join('\t', champs);
    File.WriteAllLines(Path.Combine(trafiqueDir, "derivations.tsv"), lignes, new UTF8Encoding(false));
    var resForge = CheckerTsv.VerifierOoCert(Path.Combine(trafiqueDir, "asserted.tsv"),
                                             Path.Combine(trafiqueDir, "derivations.tsv"), ooRules);
    // Résumé INVARIANT de la forge E : ce que toute exécution doit donner.
    // total_problems, lui, est une MESURE DE RUN : l'ordre des dérivations du
    // binaire n'est pas stable (constaté firsthand : 2 problèmes sur un run, 1 sur un autre).
    bool rejete = (int)resForge["total_problems"]! >= 1;
    bool aCm = ((List<string>)resForge["reasons"]!).Contains("CONCLUSION_MISMATCH");
    Console.WriteLine("Forge E — certificat réel trafiqué — résumé invariant :");
    Console.WriteLine($"  rejet                            : {rejete}");
    Console.WriteLine($"  contient CONCLUSION_MISMATCH     : {aCm}");
    Console.WriteLine($"  total_problems (mesure de CE run): {resForge["total_problems"]} — "
        + "variable entre exécutions, jamais une garantie");
    Console.WriteLine($"  raisons observées                : {string.Join(", ", (List<string>)resForge["reasons"]!)}");
}

Découplage vérifié : VerifierOoCert utilise LierPatronChecker, pas MatchMoteur().


Certificat réel tel qu'émis      : {"steps":258,"total_problems":0,"reasons":[],"problems":[]}


Forge E — certificat réel trafiqué — résumé invariant :


  rejet                            : True


  contient CONCLUSION_MISMATCH     : True


  total_problems (mesure de CE run): 2 — variable entre exécutions, jamais une garantie


  raisons observées                : CONCLUSION_MISMATCH, PREMISE_NOT_AVAILABLE



(9,44): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(56,45): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Interprétation — un checker de quelques dizaines de lignes face à un moteur de production

**Sortie obtenue** : le certificat émis par le binaire passe **intégralement** (258 étapes, 0 problème) ; la copie trafiquée est **rejetée**, et le rejet contient `CONCLUSION_MISMATCH` — l'étape forgée (sujet suffixé `FORGE`) ne se reconstruit depuis aucune prémisse. C'est l'**invariant** que nous clamons, et rien de plus : le **nombre** de problèmes varie d'une exécution à l'autre, car l'ordre des dérivations du binaire n'est **pas stable** (constaté firsthand : deux problèmes sur une exécution, un seul sur la suivante). Quand une dérivation aval suit l'étape empoisonnée dans l'ordre du fichier, elle tombe en `PREMISE_NOT_AVAILABLE` — une cascade **possible**, jamais garantie. C'est pourquoi la cellule imprime un résumé stable : rejet + présence de `CONCLUSION_MISMATCH`, le total étant présenté comme mesure de run, pas comme promesse.

Ce qu'il faut en retenir — et ce qu'il ne faut **pas** en conclure :

- **Ce qui est démontré** : le format certificat rend la vérification *bon marché* et *indépendante du moteur*. Le fichier ne demande ni Oxigraph, ni Rust, ni point fixe : quelques dizaines de lignes de C# sur la seule base de classes — avec leur **propre unification** — relisent les 258 dérivations, prémisses dérivées comprises (l'ordre du fichier suffit à justifier chaque chaîne).
- **Ce qui n'est PAS démontré** : notre jouet applique la *procédure* de vérification, il ne porte **aucun théorème**. Chez open-ontologies, c'est le checker Lean qui est prouvé *sound* (`OOCert.certificate_sound` : si le checker accepte, alors la conclusion est vraie dans tout modèle du graphe asserté). Notre C# n'a pas d'équivalent — c'est un instrument pédagogique, pas une jambe de confiance.

## Exemple guidé 2 — La forge de la double liaison (décision 0008)

Le README d'open-ontologies raconte une divergence réelle trouvée par son différentiel multi-kernels : deux checkers valides lisaient différemment un certificat dont une **même variable était liée à deux valeurs différentes** dans les prémisses — « *A key bound twice to different values is satisfied by no substitution at all* » (décision 0008 : *a binding is data and evidence admits one reading*). Nous forgeons cette attaque **ensemble** — l'exercice 2 ci-dessous attaquera par un autre versant : la table de règles elle-même.

(a) construire `forge2` : une copie (`CopieProfonde`) du certificat honnête, augmentée d'une étape `rdfs11` dont les prémisses lient `?b` à deux valeurs différentes — `(NamedPizza subClassOf Pizza)` et `(Food subClassOf Consumable)` — concluant `(NamedPizza subClassOf Consumable)` ;
(b) exécuter `VerifierCertificat` sur cette forge avec la claim `margherita/hasAllergen/milk` et **nommer** la raison du rejet ;
(c) expliquer en une phrase pourquoi aucune substitution ne peut sauver cette étape.

In [12]:
// Exemple guidé 2 — forge de la double liaison, rejet attendu.
// (étape construite dans une variable : l'appel `params` expansé imbriqué
//  dans Add(...) ne compile pas sous le parseur C# scripting)
var etapeDouble = Etape("rdfs11",
    (Q("NamedPizza"), RdfsSubClass, Q("Consumable")),
    (Q("NamedPizza"), RdfsSubClass, Q("Pizza")), (Q("Food"), RdfsSubClass, Q("Consumable")));
var forge2 = CopieProfonde(certificate);
forge2["derivations"]!.AsArray().Add(etapeDouble);
var verdictEx2 = CheckerCertificat.VerifierCertificat(forge2, claimAllergene, builtinRules);
Console.WriteLine($"verdict = {((string)verdictEx2["verdict"]!).ToUpper()}");
foreach (var p in (List<string>)verdictEx2["problems"]!)
    Console.WriteLine($"  REJET — {p}");

verdict = REJECTED


  REJET — étape 9 [rdfs11] : INCONSISTENT_BINDING


### Lecture — Exemple guidé 2 : zéro lecture, pas deux

Le rejet est `INCONSISTENT_BINDING` : la première prémisse lie `?b` à `Pizza`, la seconde à `Consumable` — aucune substitution unique ne satisfait les deux patrons à la fois, donc l'étape n'est satisfaite **par aucune lecture**. C'est exactement la classe de défaut que la décision 0008 a fermée côté format : quand une liaison est ambiguë, ce n'est pas « deux lectures possibles », c'est **zéro** lecture — et le certificat doit être refusé, pas interprété.

## Exercice 2 — La règle inconnue : faire entrer une règle par la fenêtre

**Contexte** : l'exemple guidé 2 a attaqué une **étape** (liaison incohérente). Une autre voie pour un forgeur : la **référence de règle**. La table de règles vit dans le certificat — le checker ne connaît que ce qu'elle contient.

**Objectif** :

(a) `forge3` : copie de `certificate` augmentée d'une étape citant la règle `"rdfs5"` — vérifiez d'abord son absence : les clés de `certificate["rule_table"]`. Prémisses `(NamedPizza subClassOf Pizza)` et `(Pizza subClassOf Food)` (toutes deux disponibles : l'une assertée, l'autre dérivée), conclusion `(NamedPizza subClassOf Food)`. Vérifier la claim `margherita/hasAllergen/milk` : verdict et raison nommée ? Observez aussi `claim_supported` — et expliquez pourquoi la claim honnête reste « soutenue » dans un certificat pourtant rejeté ;
(b) `forge4` : `forge3`, mais avec `"rdfs5"` **ajoutée à la table** (patron de transitivité : `(?a subClassOf ?b) ; (?b subClassOf ?c) => (?a subClassOf ?c)`). Vérifier la claim `(NamedPizza subClassOf Food)` : verdict et `rules_used` — pourquoi le verdict n'est-il **pas** `entailed`, alors même qu'une dérivation honnête `rdfs11` conclut le même triplet ?
(c) toujours sur `forge4`, vérifier la claim `(NamedPizza subClassOf Consumable)` — une claim que l'étape forgée ne touche pas. Puis conclure en une phrase : une règle ajoutée au certificat peut-elle **élever** une claim vers `entailed`, ou seulement la **dégrader** ?

> **Indice :** pour (a), que fait le checker quand la règle de l'étape n'est pas une clé de `rule_table` ? Pour (b), regardez `RulesAtteintes` : la claim est conclue par DEUX étapes (l'honnête `rdfs11` et la forgée `rdfs5`) — que fait l'union des règles utilisées ?
- **Etape 1 :** `forge3 = CopieProfonde(certificate)` ; construisez l'étape dans une variable (`var etapeF = Etape("rdfs5", ...)`) puis `forge3["derivations"]!.AsArray().Add(etapeF)` — un appel `params` expansé imbriqué dans un autre appel ne compile pas en C# scripting
- **Etape 2 :** verdict sur `claimAllergene` — lire problems ET claim_supported
- **Etape 3 :** `forge4["rule_table"]!["rdfs5"] = ...` (patrons en `JsonArray`) ; verdict + rules_used
- **Etape 4 :** claim non touchée sur `forge4`, puis formuler (c)

In [13]:
// Exercice 2 — à compléter
// Etape 1 : forge3 (règle citée mais absente de la table)
// Etape 2 : verdict sur claimAllergene — problems ET claim_supported
// Etape 3 : forge4 (règle AUSSI ajoutée à la table) — verdict + rules_used
// Etape 4 : claim non touchée sur forge4, réponse à (c)
object resultatEx2 = null;  // TODO étudiant : pourquoi forge4 ne peut pas obtenir `entailed`
Console.WriteLine("Exercice 2 à compléter");

Exercice 2 à compléter


## Exemple guidé 3 — Le slice minimal : ce qu'un retranchement préserve

Le README d'open-ontologies contient une ligne qui se relit deux fois : une *retrieval slice* à 99 % de couverture peut avoir perdu **le seul triplet** dont une réponse dépend — et une à 60 % peut préserver chaque claim qui compte. La couverture est un proxy ; la propriété, c'est la **préservation d'entaillement**, décidable ici, certificat à l'appui. Nous construisons **ensemble** le slice minimal — l'exercice 3 ci-dessous en jouera le miroir : non plus extraire le nécessaire, mais retirer l'essentiel et mesurer ce qui meurt.

**Démarche guidée** : écrire `CertificatMinimal(cert, claim)` qui extrait du certificat complet le **plus petit sous-certificat** soutenant la claim :

(a) marcher en arrière depuis la claim : quelles étapes (transitivement) la soutiennent, et quels faits assertés utilisent-elles ;
(b) reconstruire un certificat ne contenant que ces étapes (dans l'ordre original) et ces faits ;
(c) vérifier que le sous-certificat **passe** le checker sur la claim, et qu'il est **minimal** : retirer n'importe quelle étape restante fait échouer la claim.

In [14]:
// Exemple guidé 3 — slice minimal et test de minimalité.
static JsonObject CertificatMinimal(JsonObject cert, (string S, string P, string O) claim) {
    // Marche arrière : étapes et faits assertés soutenant (transitivement) la claim.
    var derivationsArr = cert["derivations"]!.AsArray();
    var conclVersIdx = new Dictionary<(string, string, string), List<int>>();
    for (int i = 0; i < derivationsArr.Count; i++) {
        var c = CheckerCertificat.VersTuple(derivationsArr[i]!["conclusion"]!.AsArray());
        if (!conclVersIdx.TryGetValue(c, out var l)) { l = new(); conclVersIdx[c] = l; }
        l.Add(i);
    }
    var neededSteps = new HashSet<int>();
    var neededAsserted = new HashSet<(string, string, string)>();
    var assertedSet = cert["asserted"]!.AsArray()
        .Select(a => CheckerCertificat.VersTuple(a!.AsArray())).ToHashSet();
    var stack = new Stack<(string, string, string)>();
    stack.Push((claim.S, claim.P, claim.O));
    var seen = new HashSet<(string, string, string)>();
    while (stack.Count > 0) {
        var t = stack.Pop();
        if (!seen.Add(t)) continue;
        if (assertedSet.Contains(t)) { neededAsserted.Add(t); continue; }
        foreach (var i in conclVersIdx.GetValueOrDefault(t, new List<int>())) {
            neededSteps.Add(i);
            foreach (var p in derivationsArr[i]!["premises"]!.AsArray())
                stack.Push(CheckerCertificat.VersTuple(p!.AsArray()));
        }
    }
    return new JsonObject {
        ["format"] = cert["format"]!.GetValue<string>() + "/slice",
        ["rule_table"] = cert["rule_table"]!.DeepClone()!,
        ["asserted"] = new JsonArray(cert["asserted"]!.AsArray()
            .Where(a => neededAsserted.Contains(CheckerCertificat.VersTuple(a!.AsArray())))
            .Select(a => (JsonNode?)a.DeepClone()).ToArray()),
        ["derivations"] = new JsonArray(neededSteps.OrderBy(i => i)
            .Select(i => (JsonNode?)derivationsArr[i]!.DeepClone()).ToArray()),
    };
}

static JsonObject Ampute(JsonObject sous, int k) {
    var derivs = sous["derivations"]!.AsArray();
    return new JsonObject {
        ["format"] = sous["format"]!.GetValue<string>(),
        ["rule_table"] = sous["rule_table"]!.DeepClone()!,
        ["asserted"] = sous["asserted"]!.DeepClone()!,
        ["derivations"] = new JsonArray(Enumerable.Range(0, derivs.Count)
            .Where(j => j != k).Select(j => (JsonNode?)derivs[j]!.DeepClone()).ToArray()),
    };
}

var sousCert = CertificatMinimal(certificate, claimType);
var verdictSous = CheckerCertificat.VerifierCertificat(sousCert, claimType, builtinRules);
Console.WriteLine($"Sous-certificat : {sousCert["asserted"]!.AsArray().Count} assertés, "
    + $"{sousCert["derivations"]!.AsArray().Count} étapes (sur {certificate["derivations"]!.AsArray().Count})");
Console.WriteLine($"Verdict sur la claim : {verdictSous["verdict"]}  "
    + $"(règles : {string.Join(", ", (List<string>)verdictSous["rules_used"]!)})");

// Minimalité : retirer chaque étape restante doit casser la claim.
bool minimal = Enumerable.Range(0, sousCert["derivations"]!.AsArray().Count)
    .All(k => !(bool)CheckerCertificat.VerifierCertificat(
        Ampute(sousCert, k), claimType, builtinRules)["claim_supported"]!);
Console.WriteLine($"Minimalité (retrait de chaque étape casse la claim) : {minimal}");

Sous-certificat : 4 assertés, 3 étapes (sur 9)


Verdict sur la claim : entailed  (règles : rdfs11, rdfs9)


Minimalité (retrait de chaque étape casse la claim) : True



(33,35): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(35,35): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(46,54): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Lecture — Exemple guidé 3 : trois étapes sur neuf suffisent

Pour la claim `margherita/type/Consumable`, le slice complet tombe à **3 étapes sur 9** (et 4 faits assertés sur 9) : l'étape `rdfs9` finale, l'étape `rdfs9` qui établit `margherita/type/Pizza`, et l'étape `rdfs11` qui établit `Pizza/subClassOf/Consumable`. Le retrait de n'importe laquelle casse la claim — c'est la définition opérationnelle d'un slice **qui préserve**, par opposition à un slice qui ne fait que couvrir. open-ontologies pousse cette idée jusqu'au théorème de localité (Cuenca Grau, Horrocks, Kazakov et Sattler, JAIR 31, 2008) pour ses modules d'ontologie — un théorème **cité**, pas machine-checké : voir la section 7.

## Exercice 3 — L'amputation : validité globale du certificat vs support local d'une claim

**Contexte** : le slice minimal de l'exemple guidé 3 est **constructif** — il ne garde que le nécessaire. Le miroir **destructif** est tout aussi instructif : retirer une seule étape du certificat honnête et mesurer quelles claims perdent leur support. Et la sortie réserve une distinction que le checker code en dur : **le verdict porte sur le certificat entier, `claim_supported` porte sur la claim seule** — deux questions différentes.

**Objectif** : construire `certAmpute`, copie de `certificate` **sans** l'étape `rdfs11` qui conclut `(Pizza subClassOf Consumable)` — trouvez-la par son **contenu**, pas par son indice (les indices se décalent après retrait).

(a) vérifier la claim `margherita/type/Consumable` (`claimType`) sur `certAmpute` : verdict, raisons nommées (`PREMISE_NOT_AVAILABLE`), `claim_supported` — combien d'étapes cassent, et que avaient-elles en commun ?
(b) vérifier la claim `margherita/hasAllergen/milk` (`claimAllergene`) sur `certAmpute` : verdict — et `claim_supported` ? Mesuré exactement : la claim garde son support (`claim_supported: true`) dans un certificat pourtant `rejected`. Expliquez pourquoi le checker est **fail-closed** : n'importe quel problème, même sans rapport avec la claim, rejette le fichier entier ;
(c) calculer la couverture : `certAmpute["derivations"].Count / certificate["derivations"].Count` — puis formuler en une phrase pourquoi un pourcentage de couverture ne prédit ni la mort ni la survie d'une claim (reliez à la *retrieval slice* du README citée en exemple guidé 3).

> **Indice :** pour (a), listez les dérivations dont une prémisse est `(Pizza, subClassOf, Consumable)`. Pour (b), relisez le régime du checker : un problème quelconque => verdict rejected pour le FICHIER ; claim_supported dit si la claim, elle, tenait sans l'étape retirée.
- **Etape 1 :** cible = `(Q("Pizza"), RdfsSubClass, Q("Consumable"))` ; `certAmpute` = copie de `certificate` sans l'étape qui la conclut
- **Etape 2 :** `VerifierCertificat(certAmpute, claimType, builtinRules)` — verdict / problems / claim_supported
- **Etape 3 :** `VerifierCertificat(certAmpute, claimAllergene, builtinRules)` — verdict / claim_supported, expliquer le fail-closed
- **Etape 4 :** couverture = dérivations de `certAmpute` / dérivations de `certificate` ; réponse à (c)

In [15]:
// Exercice 3 — à compléter
// Etape 1 : certAmpute — retrait de l'étape concluant (Pizza, subClassOf, Consumable)
// Etape 2 : claimType sur certAmpute — problems (combien d'étapes cassent ?)
// Etape 3 : claimAllergene sur certAmpute — verdict vs claim_supported (fail-closed)
// Etape 4 : couverture et réponse à (c)
object resultatEx3 = null;  // TODO étudiant : (couverture, pourquoi type meurt mais pas le support allergène)
Console.WriteLine("Exercice 3 à compléter");

Exercice 3 à compléter


## 7. Ce que le certificat NE prouve PAS

Le README d'open-ontologies consacre une section entière à ce qu'il ne prouve pas (« *What is NOT proved* ») et la place **dans le README plutôt que dans un fichier que personne n'ouvre** : « *Every line above is worth less if this section is missing* ». C'est une leçon de design en soi. Les limites ci-dessous sont celles qui portent le plus — vérifiées à la source au tag v1.4.0 :

| Limite | Ce qu'elle signifie | Source |
|---|---|---|
| **La provenance des assertions** | le certificat dit « SI le graphe asserté est bien ce qu'il prétend, ALORS… ». Rien ne lie le graphe exporté au graphe raisonné : exporter A, raisonner sur B, et le checker accepte. L'issue #158 du dépôt (« *certified input: the reasoner takes a value, not a store* », décision 0010) documente le cas **réalisé** d'une réfutation valide prouvant l'inconsistance d'« une ontologie sur deux versions qui n'ont jamais coexisté » — *the proof was correct; the graph was wrong* | issue #158 (ouverte au tag v1.4.0) |
| **Deux moteurs, un seul algorithme** | le moteur Rust et le moteur Python pur (`open-ontologies-lite`) exécutent **la même table de règles** ; leur accord est une preuve contre l'erreur de transcription, quasi nulle contre une mélecture partagée d'une règle W3C. La jambe indépendante est le **checker Lean** | README v1.4.0, « *What is in the box* » |
| **Le théorème de localité est cité, pas machine-checké** | l'extraction de modules (`onto_module_extract`) repose sur le théorème de couverture de Cuenca Grau, Horrocks, Kazakov et Sattler (JAIR 31, 2008). Le rapport **nomme le papier et jamais un théorème Lean**, porte un champ `guarantee_is_not_machine_checked`, et un test (`the_module_report_never_names_a_lean_theorem`) l'exige | `docs/decisions/0011`, `docs/modules-and-conservativity.md` |
| **Le moteur Rust est presque entièrement non vérifié** | sérialiseurs, parseurs, interner : tout ce qui est « à gauche » du certificat est testé par propriétés, borné par model-checking partiel — pas prouvé. Les théorèmes sont **conditionnels** | README v1.4.0, `docs/trusted-computing-base.md` |
| **Les réponses négatives sont des opinions** | une réfutation ne peut pas être rejouée dans core Lean : un « insatisfiable » est un témoignage d'oracle, pas un certificat. Le vocabulaire distingue systématiquement le **mesuré** du **prouvé** (décision 0006) | README v1.4.0 |

Appliqué à notre jouet : le checker de la section 4 vérifie une *procédure*, sans théorème ; il ne sait rien de la provenance des 9 faits assertés ; et si nous écrivions une seconde implémentation du moteur en copiant la première, leur accord ne prouverait rien — c'est le checker qui doit être indépendant, pas le moteur duplicata.

## 8. L'honnêteté documentaire comme discipline

Les documents d'un projet peuvent dériver de son code sans qu'aucun fichier ne soit faux pris isolément : chaque page recopie un nombre qui, lui, a changé. Le cas d'école vient du dépôt open-ontologies, dont la documentation annonçait des comptes d'outils et de fonctionnalités qui n'étaient plus ceux du binaire livré. La campagne d'audit qui a suivi se résume d'une phrase :

L'audit a examiné dix affirmations et relevé 131 copies du même compte à travers la documentation — assez pour qu'un lecteur qui en vérifie une quelconque trouve le dépôt en contradiction avec lui-même.

Dix affirmations, 131 occurrences, un solde de lignes à sens unique. Les écarts relevés sont de la matière première pour un cours sur les claims :

| Claim avant | Vérité mesurée | Forme du défaut |
|---|---|---|
| « 109 tools »… ou « 70+ », « 50+ », « 103 », « 43 » selon les fichiers | 109 au moment du sweep | quatre fichiers, quatre nombres |
| « the reasoner is SHOIQ » | SHIQ — pas de nominaux, `owl:oneOf` jamais parsé | fragment survendu |
| « 0 disagreements with HermiT » | « 0 **unsound** rejections » — 23 des 78 884 paires restent au palier résiduel | le « sans désaccord » masquait une incomplétude |
| le quickstart promet un modèle d'embeddings téléchargé par `init` | réservé aux builds avec feature `embeddings` | promesse hors build publié |

Et la discipline va jusqu'aux **tests** : `no_stale_tool_count_survives_anywhere` vérifie que les comptes des docs sont **dérivés** du code, pas retapés à la main — un compteur qui dérive fait échouer la CI.

**La règle vaut pour tout exposé de mesures.** Un compte se cite avec son **tag et sa date**, jamais nu : le README de la v1.4.0 annonce 114 outils là où l'audit de septembre en comptait 109 — le nombre évolue vite, et une phrase qui le fige devient fausse sans qu'on y touche. C'est la même conclusion que celle des mainteneurs d'open-ontologies : **un claim non mesuré est une dette**.


## 9. Avec une preuve, sans une preuve — la table fil rouge, définitive

| Question | Sans preuve (SW-13 : owlrl, HermiT…) | Avec certificat — jouet (sections 2-5) | Avec certificat — open-ontologies v1.4.0 (section 6) |
|---|---|---|---|
| Pourquoi croire la conclusion ? | « le raisonneur l'a dit » | 9 étapes relues, prémisses ground | 258 étapes émises, prémisses ground |
| Qui peut vérifier ? | qui ré-exécute le moteur | tout lecteur du fichier JSON | tout lecteur des TSV — checker Lean prouvé sound (`OOCert.certificate_sound`) |
| Conclusion forgée ? | indétectable | rejetée 5 fois sur 5 (forges A-D + E) | rejetée (forge E : `CONCLUSION_MISMATCH`) |
| Règle métier ? | même statut qu'un fait | `entailed_under_supplied_rules` ≠ `entailed` | même distinction, testée par la CI |
| Le « non » (insatisfiable) ? | opinion | hors scope du jouet | opinion d'oracle — dite comme telle |
| Coût de vérification ? | ré-exécuter le raisonnement | relire le fichier | relire le fichier (+ théorème déjà payé) |

### Cross-links — où ce notebook se branche dans le dépôt

- **[SW-13-Reasoners](SW-13-Reasoners-CSharp.ipynb)** — le gap que ce notebook comble : benchmark de raisonneurs sans preuve transportable ;
- **#16751 / #16741** (vericoding vs vibe coding, Epic Tegmark) — le pont désigné : *prouver les programmes générés par LLM* côté code, *prouver les inférences d'une base de connaissances* côté données. open-ontologies incarne la boucle complète : un LLM écrit l'ontologie (plugin Claude/MCP, ~120 outils `onto_*`), le moteur certifie, Lean prouve le certificat — « l'agent écrit, la machine prouve » ;
- **[Z3-Linq2Z3](../SMT/Z3-Linq2Z3/)** — open-ontologies utilise Z3 et cvc5 comme **oracles différentiels** ; nos notebooks SMT sont le prérequis naturel du geste « croiser plusieurs vérificateurs sur le même objet » ;
- **[SW-15](SW-15-Coup-Argumentatif-Python.ipynb) / [Tweety](../Tweety/)** — les justifications minimales (prémisses qui soutiennent une conclusion, exemple guidé 3) résonnent avec les *warrants* de l'argumentation formelle : notre slice minimal est l'analogue épistémique d'un argument sans attaque ;
- **#5721** (prior art) — l'ingénierie ontologique Argumentum → CoursIA (SKOS, AIF.owl) : l'ontologie AIF de SW-15 et la validation certifiée de SW-16 sont les deux moitiés d'un même programme ;
- **[Lean](../Lean/)** — la série pousse l'idée à son terme : ici le checker est *décrit*, là-bas il est *prouvé*.

## Conclusion — le gap comblé, et ce qui reste

**Ce que vous emportez** :

1. Une conclusion de raisonneur sans preuve transportable est un **acte de confiance** ; le certificat en fait un **fait vérifiable** par un tiers qui ne fait pas confiance au moteur ;
2. Émettre la preuve coûte peu : un forward chaining naïf qui enregistre règle + conclusion + prémisses — sans beaucoup plus de code que la clôture owlrl de SW-13 ;
3. Vérifier coûte encore moins : relire chaque étape sous une seule substitution, exiger la disponibilité **ordonnée** des prémisses — et **refuser nommément** (5 familles de rejets démontrées) ;
4. Le verdict est honnête par construction : `entailed` vs `entailed_under_supplied_rules`, l'oracle distingué du certificat, les limites écrites (provenance #158, moteurs jumeaux, théorème cité non machine-checké).

**Ce qui reste ouvert** — et c'est le programme : la provenance des assertions (issue #158 ouverte au tag : le certificat le plus juste ne prouve rien si le graphe asserté ment) ; la réfutation certifiée ; et pour ce dépôt, la déclinaison Lean du checker jouet — un compagnon à kernel Lean qui compilerait `horn_certificate_sound` sur notre mini-table de règles serait le prolongement naturel (la série [Lean](../Lean/README.md) en a tous les outils).

**Ressources** : open-ontologies au tag [v1.4.0](https://github.com/fabio-rovai/open-ontologies/tree/v1.4.0) (moteur + `lean/OOCert` + `docs/decisions/`) ; Rovai, *Open Ontologies: Tool-Augmented Ontology Engineering with Stable Matching Alignment*, arXiv:2605.09184 (2026) ; Cuenca Grau, Horrocks, Kazakov et Sattler, *Modular Reuse of Ontologies: Theory and Practice*, JAIR 31, 2008 ; Necula, *Proof-Carrying Code*, POPL 1997.